# Fashion Brand Entity Recognition and Retrieval from News Articles

This project combines Natural Language Processing (NLP) and Information Retrieval (IR) techniques to identify fashion brands in news articles and retrieve relevant fashion-related content.

The project uses weak supervision to automatically generate training labels and fine-tunes a DistilBERT model for Named Entity Recognition (NER). A BM25-based retrieval system is also implemented to search and rank fashion news articles.

## Environment and Libraries

This section imports the Python libraries and frameworks used throughout the project.

In [3]:
import numpy as np
import scipy
import pandas as pd
import pyarrow as pa
import datasets
import sys
import torch
import re
import math
import transformers
from datasets import load_dataset
from datasets import Dataset, DatasetDict
from collections import Counter, defaultdict
from typing import List
from pathlib import Path
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification
from transformers import TrainingArguments
from evaluate import load
from transformers import Trainer
from transformers import pipeline
import json

In [4]:
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("numpy:", np.__version__)

torch: 2.12.0
transformers: 5.8.1
datasets: 4.8.4
numpy: 2.4.4


## Dataset Loading

The CC-News dataset is used as the source corpus for fashion-related news articles.

In [6]:
ds = load_dataset("vblagoje/cc_news", split = "train")
print("num rows:", len(ds))
print(ds.features)

num rows: 708241
{'title': Value('string'), 'text': Value('string'), 'domain': Value('string'), 'date': Value('string'), 'description': Value('string'), 'url': Value('string'), 'image_url': Value('string')}


In [7]:
for i in range(3):
    ex = ds[i]
    print("\n")
    print("DATE  :", ex["date"])
    print("DOMAIN:", ex["domain"])
    print("TITLE :", (ex["title"] or "")[:160])
    print("URL   :", ex["url"])
    text = (ex["text"] or "").replace("\n"," ")
    print("TEXT  :", text[:400], "...")



DATE  : 2017-12-11 20:19:05
DOMAIN: www.pointemagazine.com
TITLE : Daughter Duo is Dancing in The Same Company
URL   : http://www.pointemagazine.com/mother-daughter-duo-dancing-2516681965.html
TEXT  : There's a surprising twist to Regina Willoughby's last season with Columbia City Ballet: It's also her 18-year-old daughter Melina's first season with the company. Regina, 40, will retire from the stage in March, just as her daughter starts her own career as a trainee. But for this one season, they're sharing the stage together. Performing Side-By-Side In The Nutcracker Regina and Melina are not o ...


DATE  : 2017-12-11 17:02:55
DOMAIN: www.pointemagazine.com
TITLE : New York City Ballet Announces Interim Leadership Team
URL   : http://www.pointemagazine.com/nycb-interim-leadership-team-2516618703.html
TEXT  : The New York City Ballet Board of Directors announced on Saturday the interim team that has been appointed to run the artistic side of the company during ballet master in chief 

## Fashion Brand Dictionary

A dictionary of fashion brands and aliases is manually constructed to support weak supervision and entity matching.

In [9]:
BRANDS = {
    "Gucci": ["Gucci"],
    "Prada": ["Prada"],
    "Louis Vuitton": ["Louis Vuitton", "LV"],
    "Dior": ["Dior", "Christian Dior"],
    "Chanel": ["Chanel"],
    "Saint Laurent": ["Yves Saint Laurent", "Saint Laurent", "YSL"],
    "Balenciaga": ["Balenciaga"],
    "Burberry": ["Burberry"],
    "Versace": ["Versace"],
    "Bottega Veneta": ["Bottega Veneta"],
    "Fendi": ["Fendi"],
    "Celine": ["Celine", "Céline"],
    "Loewe": ["Loewe"],
    "Givenchy": ["Givenchy"],
    "Valentino": ["Valentino"],
    "Hermès": ["Hermès", "Hermes"],
    "Carolina Herrera": ["Carolina Herrera"],
    "Cartier": ["Cartier"],
    "Moncler": ["Moncler"],
    "Miu Miu": ["Miu Miu"],

    "Armani": ["Armani", "Giorgio Armani", "Emporio Armani"],
    "Dolce & Gabbana": ["Dolce & Gabbana", "Dolce and Gabbana", "D&G"],
    "Alexander McQueen": ["Alexander McQueen", "McQueen"],
    "Off-White": ["Off-White", "Off White"],
    "Jacquemus": ["Jacquemus"],
    "Maison Margiela": ["Maison Margiela", "Margiela"],
    "Kenzo": ["Kenzo"],
    "Moschino": ["Moschino"],
    "Tom Ford": ["Tom Ford"],
    "Ralph Lauren": ["Ralph Lauren", "Polo Ralph Lauren"],
    "Calvin Klein": ["Calvin Klein"],
    "Michael Kors": ["Michael Kors"],
    "Marc Jacobs": ["Marc Jacobs"],
    "Jimmy Choo": ["Jimmy Choo"],
    "Ferragamo": ["Ferragamo", "Salvatore Ferragamo"],
    "Oscar de la Renta": ["Oscar de la Renta"],
    "Elie Saab": ["Elie Saab"],
    "Lanvin": ["Lanvin"],
    "Balmain": ["Balmain"],
    "Etro": ["Etro"],
    "Brunello Cucinelli": ["Brunello Cucinelli"],
    "Loro Piana": ["Loro Piana"],
    "Zegna": ["Zegna", "Ermenegildo Zegna"],
    "Emilio Pucci": ["Emilio Pucci", "Pucci"],
    "Alaïa": ["Alaïa", "Alaia"],
    "Mugler": ["Mugler", "Thierry Mugler"],
    "Schiaparelli": ["Schiaparelli"],
    "Jean Paul Gaultier": ["Jean Paul Gaultier", "Gaultier"],
    "Vivienne Westwood": ["Vivienne Westwood"],
    "Issey Miyake": ["Issey Miyake"],
    "Comme des Garçons": ["Comme des Garçons", "Comme des Garcons"],
    "Dries Van Noten": ["Dries Van Noten"],
    "Rick Owens": ["Rick Owens"],
    "Acne Studios": ["Acne Studios"],
    "Vetements": ["Vetements"],
    "The Row": ["The Row"],
    "Toteme": ["Toteme", "Totême"],

    "Tiffany & Co.": ["Tiffany & Co.", "Tiffany"],
    "Rolex": ["Rolex"],
    "Omega": ["Omega"],
    "Patek Philippe": ["Patek Philippe"],
    "Audemars Piguet": ["Audemars Piguet"],
    "Tag Heuer": ["Tag Heuer"],
    "Breitling": ["Breitling"],
    "Bulgari": ["Bulgari", "BVLGARI"],
    "Van Cleef & Arpels": ["Van Cleef & Arpels"],
    "Chopard": ["Chopard"],
    "Piaget": ["Piaget"],

    "Nike": ["Nike"],
    "Adidas": ["Adidas"],
    "Puma": ["Puma"],
    "New Balance": ["New Balance"],
    "Reebok": ["Reebok"],
    "Under Armour": ["Under Armour"],
    "Asics": ["Asics"],
    "Converse": ["Converse"],
    "Vans": ["Vans"],
    "Air Jordan": ["Air Jordan"],
    "Stussy": ["Stussy", "Stüssy"],
    "Fear of God": ["Fear of God"],
    "A Bathing Ape": ["A Bathing Ape", "BAPE"],
    "Stone Island": ["Stone Island"],
    "Palm Angels": ["Palm Angels"],

    "Zara": ["Zara"],
    "H&M": ["H&M", "H and M"],
    "Uniqlo": ["Uniqlo"],
    "Mango": ["Mango"],
    "Massimo Dutti": ["Massimo Dutti"],
    "Pull & Bear": ["Pull & Bear"],
    "Bershka": ["Bershka"],
    "Stradivarius": ["Stradivarius"],
    "Shein": ["Shein"],
    "Forever 21": ["Forever 21"]
}

In [10]:
#trying to match gucci GUCCI , Hermès hermes,allow flexible spaces between words
def surface_to_regex(surface: str) -> str:
  
    s = re.escape(surface)
    s = s.replace(r"\ ", r"\s+")
    return rf"(?<!\w){s}(?!\w)"
    
brand_patterns = {}
for canon, surfaces in BRANDS.items():
    pieces = [surface_to_regex(s) for s in surfaces]
    pattern = re.compile(r"(?:%s)" % "|".join(pieces), flags=re.IGNORECASE)
    brand_patterns[canon] = pattern

list(brand_patterns.keys())[:]  # quick peek


['Gucci',
 'Prada',
 'Louis Vuitton',
 'Dior',
 'Chanel',
 'Saint Laurent',
 'Balenciaga',
 'Burberry',
 'Versace',
 'Bottega Veneta',
 'Fendi',
 'Celine',
 'Loewe',
 'Givenchy',
 'Valentino',
 'Hermès',
 'Carolina Herrera',
 'Cartier',
 'Moncler',
 'Miu Miu',
 'Armani',
 'Dolce & Gabbana',
 'Alexander McQueen',
 'Off-White',
 'Jacquemus',
 'Maison Margiela',
 'Kenzo',
 'Moschino',
 'Tom Ford',
 'Ralph Lauren',
 'Calvin Klein',
 'Michael Kors',
 'Marc Jacobs',
 'Jimmy Choo',
 'Ferragamo',
 'Oscar de la Renta',
 'Elie Saab',
 'Lanvin',
 'Balmain',
 'Etro',
 'Brunello Cucinelli',
 'Loro Piana',
 'Zegna',
 'Emilio Pucci',
 'Alaïa',
 'Mugler',
 'Schiaparelli',
 'Jean Paul Gaultier',
 'Vivienne Westwood',
 'Issey Miyake',
 'Comme des Garçons',
 'Dries Van Noten',
 'Rick Owens',
 'Acne Studios',
 'Vetements',
 'The Row',
 'Toteme',
 'Tiffany & Co.',
 'Rolex',
 'Omega',
 'Patek Philippe',
 'Audemars Piguet',
 'Tag Heuer',
 'Breitling',
 'Bulgari',
 'Van Cleef & Arpels',
 'Chopard',
 'Piaget',

In [11]:
#finding set of canonical brand names
def find_brands_in_text(text: str) -> list[str]:
    if not text:
        return []
    hits = []
    for canon, pat in brand_patterns.items():
        if isinstance(pat, list):
            if any(p.search(text) for p in pat):
                hits.append(canon)
        else:
            if pat.search(text):
                hits.append(canon)
    return hits

In [12]:
# Work on a manageable slice for prototyping
ds_small = ds.shuffle(seed=42).select(range(min(100_000, len(ds))))
len(ds_small)

100000

In [13]:
def extract_brands_row(ex):
    # join title + body for detection
    text = ((ex.get("title") or "") + " " + (ex.get("text") or "")).replace("\n", " ")
    brands = find_brands_in_text(text)
    ex["brands"] = brands
    ex["hit"] = len(brands) > 0
    return ex

# tag all examples in the 100k slice.
tagged = ds_small.map(extract_brands_row)
tagged

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'text', 'domain', 'date', 'description', 'url', 'image_url', 'brands', 'hit'],
    num_rows: 100000
})

In [14]:
# Make sure every brand_patterns[canon] is a LIST of regex patterns
for canon, pat in list(brand_patterns.items()):
    if isinstance(pat, re.Pattern):
        brand_patterns[canon] = [pat]
    elif isinstance(pat, list):
        # ok already
        pass
    else:
        raise TypeError(f"brand_patterns['{canon}'] has unexpected type: {type(pat)}")

print("✅ brand_patterns normalized. Example entry type:", type(next(iter(brand_patterns.values()))), 
      "length:", len(next(iter(brand_patterns.values()))))


✅ brand_patterns normalized. Example entry type: <class 'list'> length: 1


In [15]:
selected = [i for i, ex in enumerate(tagged) if ex["hit"]]
brand_hit = tagged.select(selected)
len(brand_hit)
#each row = one full news article

1520

In [16]:
for i in range(5):
    ex = brand_hit[i]
    print("\n---")
    print("TITLE :", (ex["title"] or "")[:180])
    print("BRANDS:", ex["brands"])


---
TITLE : Trump, Macron honor 'joint history' between US, France at White House state dinner
BRANDS: ['Louis Vuitton', 'Chanel']

---
TITLE : The History of the Girl Scout Cookie
BRANDS: ['Mango']

---
TITLE : Police say southern Kentucky woman allowed boyfriend to have sex - WDRB 41 Louisville News
BRANDS: ['Tiffany & Co.']

---
TITLE : Requiem could be the BBC's most terrifying whodunnit drama ever
BRANDS: ['Converse']

---
TITLE : StockX Adds BAPE and PALACE to its 'Stock Market of Things'
BRANDS: ['A Bathing Ape']


In [17]:
cnt = Counter()
for ex in brand_hit:
    cnt.update(ex["brands"])
cnt.most_common(20)

[('Nike', 161),
 ('Tiffany & Co.', 125),
 ('Vans', 124),
 ('Omega', 100),
 ('Chanel', 82),
 ('Adidas', 78),
 ('Gucci', 73),
 ('Mango', 72),
 ('Under Armour', 72),
 ('Louis Vuitton', 67),
 ('Zara', 66),
 ('Celine', 56),
 ('The Row', 55),
 ('Alexander McQueen', 54),
 ('Dior', 49),
 ('H&M', 47),
 ('Armani', 46),
 ('Versace', 44),
 ('Puma', 39),
 ('Prada', 37)]

In [18]:
keep_cols = ["date", "domain", "url", "title", "text", "brands"]
brand_hit_small = brand_hit.remove_columns([c for c in brand_hit.column_names if c not in keep_cols])
brand_hit_small


Dataset({
    features: ['title', 'text', 'domain', 'date', 'url', 'brands'],
    num_rows: 1520
})

In [19]:
MAX_DOCS = min(5000, len(brand_hit_small))
MAX_SENTS_PER_DOC = 5
MAX_SENT_LEN = 220


In [20]:
def split_into_sentences(text: str) -> List[str]:
    text = (text or "").replace("\n", " ").strip()
    if not text:
        return []
    sents = re.split(r"(?<=[\.\!\?])\s+", text)
    sents = [s.strip() for s in sents if s.strip()]
    return sents


In [21]:
pool = []

for i in range(MAX_DOCS):
    ex = brand_hit_small[i]
    text = ((ex.get("title") or "") + ". " + (ex.get("text") or "")).replace("\n", " ")
    sents = split_into_sentences(text)

    added = 0
    for s in sents:
        if len(s) > MAX_SENT_LEN:
            continue

        hits = find_brands_in_text(s)
        if len(hits) == 0:
            continue

        pool.append({
            "date": ex["date"],
            "domain": ex["domain"],
            "url": ex["url"],
            "sentence": s,
            "brands_hint": ",".join(hits)
        })

        added += 1
        if added >= MAX_SENTS_PER_DOC:
            break

len(pool)


2074

In [22]:
seen = set()
unique_pool = []

for row in pool:
    key = row["sentence"].lower()
    if key not in seen:
        seen.add(key)
        unique_pool.append(row)

len(unique_pool)
#list of dictionaries where each dictionary = one sentence

1852

In [23]:
for i in range(5):
    print("\n---")
    print("HINT:", unique_pool[i]["brands_hint"])
    print("SENT:", unique_pool[i]["sentence"])



---
HINT: Chanel
SENT: Melania Trump wore what the White House described as a black Chantilly lace Chanel haute couture gown, hand-painted with silver and embroidered with crystal and sequins.

---
HINT: Louis Vuitton
SENT: Brigitte Macron wore a cream full-length gown by Louis Vuitton with long sleeves and gold details.

---
HINT: Mango
SENT: In that time, different cookies have come and gone, some fading into obscurity (anybody remember the Mango Cremes?

---
HINT: Tiffany & Co.
SENT: State troopers arrested Tiffany Walsh Thursday morning in Breathitt County, but the charges against her stem from Perry County.

---
HINT: Converse
SENT: ‘She’s a girl from London in Converse trainers whose life suddenly spirals into this giant mystery.


In [24]:
df_ann = pd.DataFrame(unique_pool)
df_ann.head()


,date,domain,url,sentence,brands_hint
0,2018-04-24 00:00:00,590kid.com,http://590kid.com/trump-macron-honor-joint-his...,Melania Trump wore what the White House descri...,Chanel
1,2018-04-24 00:00:00,590kid.com,http://590kid.com/trump-macron-honor-joint-his...,Brigitte Macron wore a cream full-length gown ...,Louis Vuitton
2,2018-02-03 15:45:40,highschool.latimes.com,http://highschool.latimes.com/la-canada-high-s...,"In that time, different cookies have come and ...",Mango
3,2017-10-06 00:00:00,www.wdrb.com,http://www.wdrb.com/story/36539027/police-say-...,State troopers arrested Tiffany Walsh Thursday...,Tiffany & Co.
4,2018-02-02 15:38:00,www.mirror.co.uk,https://www.mirror.co.uk/tv/tv-news/requiem-bb...,‘She’s a girl from London in Converse trainers...,Converse


In [25]:
out_dir = Path("annotations")
out_dir.mkdir(exist_ok=True)

out_path = out_dir / "to_annotate_v1.csv"
df_ann.to_csv(out_path, index=False)

out_path


PosixPath('annotations/to_annotate_v1.csv')

## Weak Supervision and Brand Matching

Weak supervision is used to automatically identify fashion brands in news text using regex-based matching rules.

In [27]:
def find_brand_char_spans(sentence: str):
    spans = []
    for canon, pat in brand_patterns.items():
        if isinstance(pat, list):
            patterns = pat
        else:
            patterns = [pat]

        for rx in patterns:
            for m in rx.finditer(sentence):
                spans.append({"canon": canon, "start": m.start(), "end": m.end()})

    spans.sort(key=lambda x: (x["start"], x["end"]))
    return spans


In [28]:
sent0 = df_ann.iloc[0]["sentence"]
spans0 = find_brand_char_spans(sent0)

print(sent0)
print("\nSpans:")
for sp in spans0:
    print(sp, "->", sent0[sp["start"]:sp["end"]])


Melania Trump wore what the White House described as a black Chantilly lace Chanel haute couture gown, hand-painted with silver and embroidered with crystal and sequins.

Spans:
{'canon': 'Chanel', 'start': 76, 'end': 82} -> Chanel


In [29]:
def show_with_indices(text: str, width: int = 100):
    for i in range(0, len(text), width):
        chunk = text[i:i+width]
        ruler = "".join(str((i+j) % 10) for j in range(len(chunk)))
        print(ruler)
        print(chunk)
        print()


In [30]:
show_with_indices(sent0)


0123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890123456789
Melania Trump wore what the White House described as a black Chantilly lace Chanel haute couture gow

012345678901234567890123456789012345678901234567890123456789012345678
n, hand-painted with silver and embroidered with crystal and sequins.



In [31]:
def tokenize_with_spans(sentence: str):
    tokens = []
    for m in re.finditer(r"\w+|[^\w\s]", sentence, flags=re.UNICODE):
        tokens.append({"text": m.group(0), "start": m.start(), "end": m.end()})
    return tokens


In [32]:
tokens0 = tokenize_with_spans(sent0)
[(t["text"], t["start"], t["end"]) for t in tokens0]


[('Melania', 0, 7),
 ('Trump', 8, 13),
 ('wore', 14, 18),
 ('what', 19, 23),
 ('the', 24, 27),
 ('White', 28, 33),
 ('House', 34, 39),
 ('described', 40, 49),
 ('as', 50, 52),
 ('a', 53, 54),
 ('black', 55, 60),
 ('Chantilly', 61, 70),
 ('lace', 71, 75),
 ('Chanel', 76, 82),
 ('haute', 83, 88),
 ('couture', 89, 96),
 ('gown', 97, 101),
 (',', 101, 102),
 ('hand', 103, 107),
 ('-', 107, 108),
 ('painted', 108, 115),
 ('with', 116, 120),
 ('silver', 121, 127),
 ('and', 128, 131),
 ('embroidered', 132, 143),
 ('with', 144, 148),
 ('crystal', 149, 156),
 ('and', 157, 160),
 ('sequins', 161, 168),
 ('.', 168, 169)]

## BIO Annotation

The BIO tagging scheme is used for Named Entity Recognition:

- B-BRAND: beginning of a brand entity
- I-BRAND: continuation of a brand entity
- O: non-entity token

In [34]:
def spans_to_bio_tags(tokens, spans):
    tags = ["O"] * len(tokens)

    for sp in spans:
        entity_token_indices = []

        for i, tok in enumerate(tokens):
            if tok["start"] < sp["end"] and tok["end"] > sp["start"]:
                entity_token_indices.append(i)

        if entity_token_indices:
            tags[entity_token_indices[0]] = "B-BRAND"
            for idx in entity_token_indices[1:]:
                tags[idx] = "I-BRAND"

    return tags

In [35]:
tags0 = spans_to_bio_tags(tokens0, spans0)

list(zip([t["text"] for t in tokens0], tags0))

[('Melania', 'O'),
 ('Trump', 'O'),
 ('wore', 'O'),
 ('what', 'O'),
 ('the', 'O'),
 ('White', 'O'),
 ('House', 'O'),
 ('described', 'O'),
 ('as', 'O'),
 ('a', 'O'),
 ('black', 'O'),
 ('Chantilly', 'O'),
 ('lace', 'O'),
 ('Chanel', 'B-BRAND'),
 ('haute', 'O'),
 ('couture', 'O'),
 ('gown', 'O'),
 (',', 'O'),
 ('hand', 'O'),
 ('-', 'O'),
 ('painted', 'O'),
 ('with', 'O'),
 ('silver', 'O'),
 ('and', 'O'),
 ('embroidered', 'O'),
 ('with', 'O'),
 ('crystal', 'O'),
 ('and', 'O'),
 ('sequins', 'O'),
 ('.', 'O')]

In [36]:
ner_rows = []

for i, row in df_ann.iterrows():
    sentence = row["sentence"]

    tokens = tokenize_with_spans(sentence)
    spans = find_brand_char_spans(sentence)
    tags = spans_to_bio_tags(tokens, spans)

    ner_rows.append({
        "sentence": sentence,
        "tokens": [t["text"] for t in tokens],
        "ner_tags": tags,
        "brands_hint": row["brands_hint"]
    })

len(ner_rows)

1852

In [37]:
for tok, tag in zip(ner_rows[0]["tokens"], ner_rows[0]["ner_tags"]):
    print(f"{tok:20s} {tag}")

Melania              O
Trump                O
wore                 O
what                 O
the                  O
White                O
House                O
described            O
as                   O
a                    O
black                O
Chantilly            O
lace                 O
Chanel               B-BRAND
haute                O
couture              O
gown                 O
,                    O
hand                 O
-                    O
painted              O
with                 O
silver               O
and                  O
embroidered          O
with                 O
crystal              O
and                  O
sequins              O
.                    O


In [38]:
label2id = {
    "O": 0,
    "B-BRAND": 1,
    "I-BRAND": 2
}

id2label = {
    0: "O",
    1: "B-BRAND",
    2: "I-BRAND"
}

In [39]:
for row in ner_rows:
    row["ner_tag_ids"] = [label2id[tag] for tag in row["ner_tags"]]

ner_rows[0]

{'sentence': 'Melania Trump wore what the White House described as a black Chantilly lace Chanel haute couture gown, hand-painted with silver and embroidered with crystal and sequins.',
 'tokens': ['Melania',
  'Trump',
  'wore',
  'what',
  'the',
  'White',
  'House',
  'described',
  'as',
  'a',
  'black',
  'Chantilly',
  'lace',
  'Chanel',
  'haute',
  'couture',
  'gown',
  ',',
  'hand',
  '-',
  'painted',
  'with',
  'silver',
  'and',
  'embroidered',
  'with',
  'crystal',
  'and',
  'sequins',
  '.'],
 'ner_tags': ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-BRAND',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 'brands_hint': 'Chanel',
 'ner_tag_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]}

In [40]:
df_ner = pd.DataFrame(ner_rows)

df_ner.head()

,sentence,tokens,ner_tags,brands_hint,ner_tag_ids
0,Melania Trump wore what the White House descri...,"[Melania, Trump, wore, what, the, White, House...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-BRAN...",Chanel,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
1,Brigitte Macron wore a cream full-length gown ...,"[Brigitte, Macron, wore, a, cream, full, -, le...","[O, O, O, O, O, O, O, O, O, O, B-BRAND, I-BRAN...",Louis Vuitton,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, ..."
2,"In that time, different cookies have come and ...","[In, that, time, ,, different, cookies, have, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Mango,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,State troopers arrested Tiffany Walsh Thursday...,"[State, troopers, arrested, Tiffany, Walsh, Th...","[O, O, O, B-BRAND, O, O, O, O, O, O, O, O, O, ...",Tiffany & Co.,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,‘She’s a girl from London in Converse trainers...,"[‘, She, ’, s, a, girl, from, London, in, Conv...","[O, O, O, O, O, O, O, O, O, B-BRAND, O, O, O, ...",Converse,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ..."


In [41]:
for token, tag, tag_id in zip(
    df_ner.iloc[0]["tokens"],
    df_ner.iloc[0]["ner_tags"],
    df_ner.iloc[0]["ner_tag_ids"]
):
    print(f"{token:20s} {tag:10s} {tag_id}")

Melania              O          0
Trump                O          0
wore                 O          0
what                 O          0
the                  O          0
White                O          0
House                O          0
described            O          0
as                   O          0
a                    O          0
black                O          0
Chantilly            O          0
lace                 O          0
Chanel               B-BRAND    1
haute                O          0
couture              O          0
gown                 O          0
,                    O          0
hand                 O          0
-                    O          0
painted              O          0
with                 O          0
silver               O          0
and                  O          0
embroidered          O          0
with                 O          0
crystal              O          0
and                  O          0
sequins              O          0
.             

In [42]:
out_dir = Path("annotations")
out_dir.mkdir(exist_ok=True)

ner_out_path = out_dir / "ner_dataset_v1.csv"

df_ner.to_csv(ner_out_path, index=False)

ner_out_path

PosixPath('annotations/ner_dataset_v1.csv')

In [43]:
tag_counter = Counter()

for tags in df_ner["ner_tags"]:
    tag_counter.update(tags)

tag_counter

Counter({'O': 45900, 'B-BRAND': 2117, 'I-BRAND': 622})

In [44]:
num_brand_entities = 0

for tags in df_ner["ner_tags"]:
    num_brand_entities += tags.count("B-BRAND")

num_brand_entities

2117

## Dataset Preparation

The annotated dataset is split into training and testing subsets and converted into Hugging Face Dataset format.

In [46]:
train_df = df_ner.sample(frac=0.8, random_state=42)
test_df = df_ner.drop(train_df.index)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

len(train_df), len(test_df)

(1482, 370)

In [47]:
train_df.iloc[0]

sentence       Second Nike executive leaves in wake of workpl...
tokens         [Second, Nike, executive, leaves, in, wake, of...
ner_tags           [O, B-BRAND, O, O, O, O, O, O, O, O, O, O, O]
brands_hint                                                 Nike
ner_tag_ids              [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Name: 0, dtype: object

In [48]:
train_path = out_dir / "train_ner_v1.csv"
test_path = out_dir / "test_ner_v1.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

train_path, test_path

(PosixPath('annotations/train_ner_v1.csv'),
 PosixPath('annotations/test_ner_v1.csv'))

In [49]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

dataset_dict = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['sentence', 'tokens', 'ner_tags', 'brands_hint', 'ner_tag_ids'],
        num_rows: 1482
    })
    test: Dataset({
        features: ['sentence', 'tokens', 'ner_tags', 'brands_hint', 'ner_tag_ids'],
        num_rows: 370
    })
})

In [50]:
dataset_dict["train"][0]

{'sentence': 'Second Nike executive leaves in wake of workplace complai....',
 'tokens': ['Second',
  'Nike',
  'executive',
  'leaves',
  'in',
  'wake',
  'of',
  'workplace',
  'complai',
  '.',
  '.',
  '.',
  '.'],
 'ner_tags': ['O',
  'B-BRAND',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 'brands_hint': 'Nike',
 'ner_tag_ids': [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

## Tokenization and Label Alignment

DistilBERT tokenization is applied while preserving token-level labels for Named Entity Recognition.

In [52]:
model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [53]:
tokenized_example = tokenizer(
    dataset_dict["train"][0]["tokens"],
    is_split_into_words=True,
    truncation=True
)

tokenized_example

{'input_ids': [101, 2307, 20100, 3275, 2972, 1107, 5314, 1104, 19328, 3254, 1643, 20737, 119, 119, 119, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [54]:
tokens = dataset_dict["train"][0]["tokens"]

tokenized_example = tokenizer(
    tokens,
    is_split_into_words=True,
    truncation=True
)

word_ids = tokenized_example.word_ids()

list(zip(tokenizer.convert_ids_to_tokens(tokenized_example["input_ids"]), word_ids))

[('[CLS]', None),
 ('Second', 0),
 ('Nike', 1),
 ('executive', 2),
 ('leaves', 3),
 ('in', 4),
 ('wake', 5),
 ('of', 6),
 ('workplace', 7),
 ('com', 8),
 ('##p', 8),
 ('##lai', 8),
 ('.', 9),
 ('.', 10),
 ('.', 11),
 ('.', 12),
 ('[SEP]', None)]

In [55]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True
    )

    word_ids = tokenized_inputs.word_ids()
    labels = []

    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)

        elif word_idx != previous_word_idx:
            labels.append(example["ner_tag_ids"][word_idx])

        else:
            original_label = example["ner_tag_ids"][word_idx]

            if original_label == label2id["B-BRAND"]:
                labels.append(label2id["I-BRAND"])
            else:
                labels.append(original_label)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [56]:
aligned_example = tokenize_and_align_labels(dataset_dict["train"][0])

list(zip(
    tokenizer.convert_ids_to_tokens(aligned_example["input_ids"]),
    aligned_example["labels"]
))

[('[CLS]', -100),
 ('Second', 0),
 ('Nike', 1),
 ('executive', 0),
 ('leaves', 0),
 ('in', 0),
 ('wake', 0),
 ('of', 0),
 ('workplace', 0),
 ('com', 0),
 ('##p', 0),
 ('##lai', 0),
 ('.', 0),
 ('.', 0),
 ('.', 0),
 ('.', 0),
 ('[SEP]', -100)]

In [57]:
tokenized_datasets = dataset_dict.map(tokenize_and_align_labels)

tokenized_datasets

Map:   0%|          | 0/1482 [00:00<?, ? examples/s]

Map:   0%|          | 0/370 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'tokens', 'ner_tags', 'brands_hint', 'ner_tag_ids', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1482
    })
    test: Dataset({
        features: ['sentence', 'tokens', 'ner_tags', 'brands_hint', 'ner_tag_ids', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 370
    })
})

In [58]:
tokenized_datasets = tokenized_datasets.remove_columns(
    ["sentence", "tokens", "ner_tags", "brands_hint", "ner_tag_ids"]
)

tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1482
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 370
    })
})

In [59]:
tokenized_datasets["train"][0]

{'input_ids': [101,
  2307,
  20100,
  3275,
  2972,
  1107,
  5314,
  1104,
  19328,
  3254,
  1643,
  20737,
  119,
  119,
  119,
  119,
  102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [-100, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100]}

## DistilBERT Model

A pretrained DistilBERT model is fine-tuned for token classification on the fashion brand NER task.

In [61]:
num_labels = 3

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [62]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

## Model Training

The model is trained using Hugging Face Trainer with evaluation on the test dataset.

In [64]:
training_args = TrainingArguments(
    output_dir="fashion_brand_ner_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10
)

In [65]:
seqeval = load("seqeval")

In [66]:
label_names = ["O", "B-BRAND", "I-BRAND"]

def compute_metrics(eval_preds):
    logits, labels = eval_preds

    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):

        current_preds = []
        current_labels = []

        for pred, lab in zip(prediction, label):

            if lab != -100:
                current_preds.append(label_names[pred])
                current_labels.append(label_names[lab])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [67]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [68]:
trainer.train()

/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.084972,0.063111,0.740664,0.860241,0.795987,0.980118
2,0.039569,0.034778,0.879819,0.934940,0.906542,0.990710
3,0.008746,0.032138,0.885906,0.954217,0.918794,0.990623


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=558, training_loss=0.07797185051184828, metrics={'train_runtime': 209.4788, 'train_samples_per_second': 21.224, 'train_steps_per_second': 2.664, 'total_flos': 58647970684800.0, 'train_loss': 0.07797185051184828, 'epoch': 3.0})

## Model Evaluation

Model performance is evaluated using precision, recall, F1-score, and token-level accuracy.

In [70]:
eval_results = trainer.evaluate()

eval_results

/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.008746,0.032138,3,0.885906,0.954217,0.918794,0.990623


{'eval_loss': 0.0321379080414772,
 'eval_precision': 0.8859060402684564,
 'eval_recall': 0.9542168674698795,
 'eval_f1': 0.9187935034802784,
 'eval_accuracy': 0.990623372113214}

In [71]:
final_model_path = "fashion_brand_ner_model/final"

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

final_model_path

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

'fashion_brand_ner_model/final'

## Inference Examples

The trained model is tested on custom fashion-related sentences.

In [73]:
ner_pipeline = pipeline(
    "token-classification",
    model=final_model_path,
    tokenizer=final_model_path,
    aggregation_strategy="simple"
)

test_sentence = "Gucci and Louis Vuitton launched new collections in Paris."

ner_pipeline(test_sentence)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[{'entity_group': 'BRAND',
  'score': np.float32(0.99709487),
  'word': 'Gucci',
  'start': 0,
  'end': 5},
 {'entity_group': 'BRAND',
  'score': np.float32(0.9969958),
  'word': 'Louis Vuitton',
  'start': 10,
  'end': 23}]

In [74]:
test_sentence = "Gucci and Louis Vuitton launched new collections in Paris."

preds = ner_pipeline(test_sentence)

for p in preds:
    print(p)

{'entity_group': 'BRAND', 'score': np.float32(0.99709487), 'word': 'Gucci', 'start': 0, 'end': 5}
{'entity_group': 'BRAND', 'score': np.float32(0.9969958), 'word': 'Louis Vuitton', 'start': 10, 'end': 23}


In [75]:
test_sentence = "Dior, Prada, and Saint Laurent were featured during Paris Fashion Week."

ner_pipeline(test_sentence)

[{'entity_group': 'BRAND',
  'score': np.float32(0.9950399),
  'word': 'Dior',
  'start': 0,
  'end': 4},
 {'entity_group': 'BRAND',
  'score': np.float32(0.99775743),
  'word': 'Prada',
  'start': 6,
  'end': 11},
 {'entity_group': 'BRAND',
  'score': np.float32(0.99099606),
  'word': 'Saint Laurent',
  'start': 17,
  'end': 30}]

In [76]:
with open("fashion_brand_ner_model/results.json", "w") as f:
    json.dump(eval_results, f, indent=2)

## Error Analysis

Prediction errors are analyzed to better understand the limitations of the model and the impact of weak supervision.

In [78]:
predictions_output = trainer.predict(tokenized_datasets["test"])

logits = predictions_output.predictions
labels = predictions_output.label_ids

predictions = np.argmax(logits, axis=-1)

/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [79]:
test_errors = []

for example_idx, (pred_seq, label_seq) in enumerate(zip(predictions, labels)):
    tokens = tokenizer.convert_ids_to_tokens(tokenized_datasets["test"][example_idx]["input_ids"])

    for token, pred_id, true_id in zip(tokens, pred_seq, label_seq):
        if true_id == -100:
            continue

        pred_label = id2label[int(pred_id)]
        true_label = id2label[int(true_id)]

        if pred_label != true_label:
            test_errors.append({
                "example_idx": example_idx,
                "token": token,
                "true_label": true_label,
                "pred_label": pred_label
            })

len(test_errors)

108

In [80]:
test_errors[:20]

[{'example_idx': 1, 'token': 'p', 'true_label': 'O', 'pred_label': 'B-BRAND'},
 {'example_idx': 1,
  'token': 'stock',
  'true_label': 'O',
  'pred_label': 'B-BRAND'},
 {'example_idx': 1,
  'token': '##x',
  'true_label': 'O',
  'pred_label': 'I-BRAND'},
 {'example_idx': 1, 'token': 'b', 'true_label': 'B-BRAND', 'pred_label': 'O'},
 {'example_idx': 1,
  'token': '##ap',
  'true_label': 'I-BRAND',
  'pred_label': 'O'},
 {'example_idx': 1,
  'token': '##e',
  'true_label': 'I-BRAND',
  'pred_label': 'O'},
 {'example_idx': 1,
  'token': '##x',
  'true_label': 'O',
  'pred_label': 'I-BRAND'},
 {'example_idx': 21,
  'token': 'Ba',
  'true_label': 'O',
  'pred_label': 'B-BRAND'},
 {'example_idx': 21,
  'token': '##wa',
  'true_label': 'O',
  'pred_label': 'I-BRAND'},
 {'example_idx': 53, 'token': 'D', 'true_label': 'O', 'pred_label': 'B-BRAND'},
 {'example_idx': 53,
  'token': 'Di',
  'true_label': 'O',
  'pred_label': 'I-BRAND'},
 {'example_idx': 61,
  'token': 'Air',
  'true_label': 'B-BRA

In [81]:
error_types = Counter()

for err in test_errors:
    if err["true_label"] != "O" and err["pred_label"] == "O":
        error_types["missed_entity"] += 1
    elif err["true_label"] == "O" and err["pred_label"] != "O":
        error_types["false_positive"] += 1
    else:
        error_types["boundary_or_label_error"] += 1

error_types

Counter({'false_positive': 65, 'missed_entity': 43})

In [82]:
error_df = pd.DataFrame(test_errors)

error_path = out_dir / "ner_error_analysis_v1.csv"
error_df.to_csv(error_path, index=False)

error_path

PosixPath('annotations/ner_error_analysis_v1.csv')

The detected prediction errors are exported to CSV format for further inspection and analysis.

## Information Retrieval with BM25

A BM25-based retrieval system is implemented to search and rank fashion-related news articles.

In [85]:
ir_df = df_ann.copy()

ir_df["search_text"] = (
    ir_df["sentence"].fillna("").astype(str)
    + " "
    + ir_df["brands_hint"].fillna("").astype(str)
)

len(ir_df), ir_df.head()

(1852,
                   date                  domain  \
 0  2018-04-24 00:00:00              590kid.com   
 1  2018-04-24 00:00:00              590kid.com   
 2  2018-02-03 15:45:40  highschool.latimes.com   
 3  2017-10-06 00:00:00            www.wdrb.com   
 4  2018-02-02 15:38:00        www.mirror.co.uk   
 
                                                  url  \
 0  http://590kid.com/trump-macron-honor-joint-his...   
 1  http://590kid.com/trump-macron-honor-joint-his...   
 2  http://highschool.latimes.com/la-canada-high-s...   
 3  http://www.wdrb.com/story/36539027/police-say-...   
 4  https://www.mirror.co.uk/tv/tv-news/requiem-bb...   
 
                                             sentence    brands_hint  \
 0  Melania Trump wore what the White House descri...         Chanel   
 1  Brigitte Macron wore a cream full-length gown ...  Louis Vuitton   
 2  In that time, different cookies have come and ...          Mango   
 3  State troopers arrested Tiffany Walsh Thursday...

In [86]:
def search_tokenize(text):
    text = text.lower()
    return re.findall(r"\b\w+\b", text)

In [87]:
tokenized_corpus = [search_tokenize(text) for text in ir_df["search_text"]]

doc_freq = defaultdict(int)

for tokens in tokenized_corpus:
    for token in set(tokens):
        doc_freq[token] += 1

N = len(tokenized_corpus)
avg_doc_len = sum(len(tokens) for tokens in tokenized_corpus) / N

k1 = 1.5
b = 0.75

def bm25_score(query, doc_tokens):
    query_tokens = search_tokenize(query)
    doc_counter = Counter(doc_tokens)
    doc_len = len(doc_tokens)

    score = 0.0

    for term in query_tokens:
        if term not in doc_counter:
            continue

        df = doc_freq.get(term, 0)
        idf = math.log(1 + (N - df + 0.5) / (df + 0.5))

        tf = doc_counter[term]

        numerator = tf * (k1 + 1)
        denominator = tf + k1 * (1 - b + b * doc_len / avg_doc_len)

        score += idf * numerator / denominator

    return score

In [88]:
def search_fashion_news(query, top_k=5, brand_filter=None):
    scores = []

    for i, doc_tokens in enumerate(tokenized_corpus):
        row = ir_df.iloc[i]

        if brand_filter is not None:
            brands = [b.strip().lower() for b in str(row["brands_hint"]).split(",")]
            if brand_filter.lower() not in brands:
                continue

        score = bm25_score(query, doc_tokens)

        if score > 0:
            scores.append((i, score))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

    results = []

    for i, score in scores:
        row = ir_df.iloc[i]

        results.append({
            "score": score,
            "sentence": row["sentence"],
            "brands_hint": row["brands_hint"],
            "domain": row.get("domain", ""),
            "date": row.get("date", ""),
            "url": row.get("url", "")
        })

    return results

In [89]:
results = search_fashion_news(
    query="Paris collection",
    top_k=5,
    brand_filter="Gucci"
)

for r in results:
    print("SCORE:", round(r["score"], 3))
    print("BRANDS:", r["brands_hint"])
    print("SENTENCE:", r["sentence"])
    print("URL:", r["url"])
    print("---")

SCORE: 5.098
BRANDS: Gucci
SENTENCE: This 'Gucci' collection is a fail.
URL: https://www.ibtimes.co.in/mexican-talk-show-host-says-bts-boys-look-like-lgbt-group-apologises-after-backlash-770631
---
SCORE: 4.682
BRANDS: Gucci
SENTENCE: Photo: tony gentile/Reuters PARIS—American shoppers helped Gucci start off the year with a bang.
URL: https://www.wsj.com/articles/gucci-sales-rise-on-north-american-strength-1524592112
---
SCORE: 3.812
BRANDS: Gucci
SENTENCE: A model presented at the Gucci Autumn/Winter 2018 women’s collection during Milan Fashion Week in February.
URL: https://www.wsj.com/articles/gucci-sales-rise-on-north-american-strength-1524592112
---
SCORE: 3.586
BRANDS: Gucci
SENTENCE: The shining outfit that RiRi wore first got its debut on the runway as part of Gucci’s Fall 2017 collection.
URL: http://www.inquisitr.com/4153155/rihannas-sequin-body-suit-and-glittery-face-mask-win-coachella-2017/
---
SCORE: 3.15
BRANDS: Gucci,Dior,Chanel
SENTENCE: And thus, resort was born — a co

In [90]:
def highlight_brands(sentence):
    highlighted = sentence

    spans = find_brand_char_spans(sentence)

    for sp in sorted(spans, key=lambda x: x["start"], reverse=True):
        start = sp["start"]
        end = sp["end"]
        brand_text = highlighted[start:end]
        highlighted = highlighted[:start] + "[" + brand_text + "]" + highlighted[end:]

    return highlighted

In [91]:
def show_search_results(results):
    for rank, r in enumerate(results, start=1):
        print(f"Rank {rank}")
        print("Score:", round(r["score"], 3))
        print("Brands:", r["brands_hint"])
        print("Sentence:", highlight_brands(r["sentence"]))
        print("URL:", r["url"])
        print("---")

## Retrieval Examples

Example search queries are tested using BM25 ranking and optional brand-based filtering.

In [93]:
demo_queries = [
    {
        "query": "luxury fashion Paris",
        "brand_filter": None
    },
    {
        "query": "Paris collection",
        "brand_filter": "Gucci"
    },
    {
        "query": "collection fashion week",
        "brand_filter": "Dior"
    }
]

for demo in demo_queries:
    print("=" * 80)
    print("QUERY:", demo["query"])
    print("BRAND FILTER:", demo["brand_filter"])
    print("=" * 80)

    results = search_fashion_news(
        query=demo["query"],
        top_k=5,
        brand_filter=demo["brand_filter"]
    )

    show_search_results(results)

QUERY: luxury fashion Paris
BRAND FILTER: None
Rank 1
Score: 8.465
Brands: Gucci
Sentence: “The city of Arles, also consistently involved in cultural initiatives, is happy to collaborate with a luxury brand such as [Gucci],” said the luxury fashion house in a statement.
URL: https://fashionunited.uk/news/fashion/gucci-to-host-cruise-2019-show-at-ancient-site-of-alyscamps/2017121127229
---
Rank 2
Score: 7.74
Brands: Fendi
Sentence: PARIS, July 4 (Reuters) - Italian fashion label [Fendi] played with textures in its Haute Couture collection in Paris on Wednesday, overlaying see-through bodices with fur trimmings and creating shimmering, scaly skirts.
URL: http://www.dailymail.co.uk/wires/reuters/article-5918843/Fendi-channels-shimmering-scales-latticework-catwalk-show.html?ITO=1490&ns_mchannel=rss&ns_campaign=1490
---
Rank 3
Score: 7.635
Brands: Balenciaga
Sentence: Composite: [Balenciaga]/Ikea Then there was [Balenciaga] x Crocs, released unto the world at Paris Fashion Week last year.
U

## Limitations and Future Work

The project relies on weak supervision for annotation generation, which may introduce noisy labels and evaluation bias.

Future improvements may include:
- manually annotated gold-standard datasets
- semantic retrieval methods
- multilingual fashion NER
- retrieval-augmented generation (RAG)
- integration with Large Language Models (LLMs)